# Task #6 — Customer Behavioral Drift (October Milestone 2)

**Standalone, read-only analysis notebook.** This notebook creates customer-month behavioral drift features from the **same raw Online Retail II transactions** used by `notebooks/EDA.ipynb`. It does not change the EDA, Logistic Regression, Random Forest, or XGBoost notebooks, and does not overwrite `feat.csv`.

**Deliverable:** changes in purchase frequency, revenue, unique items, average order value (AOV), and days since last purchase, plus an **initial heuristic drift score**. A high score means several indicators deteriorated; it is **not** a calibrated churn probability.

**Compatibility choices:** The monthly `month_orders`, `month_revenue`, and `month_diff_items` calculations deliberately reproduce the existing EDA's *netted customer/product/month* aggregation. This includes its known limitations (e.g., a first invoice retained per group can undercount distinct orders, and net revenue may be negative). The notebook does not silently redefine these measures. Recency follows the EDA's positive-quantity order definition and its **day-level** timestamps. The full customer-month calendar is built **before** filtering for churn labels, so prior-month comparisons remain consecutive. No future transactions are used in any feature.

**Inputs:** `data/online_retail_II.csv` from the repository or a local/Colab copy. The source EDA also reads an Excel workbook; this notebook uses the repository CSV to avoid depending on an untracked Drive workbook.

**Outputs:** in-memory `drift_features` and optional separate `behavioral_drift_features.csv`. No model retraining is performed.

## 1. Imports and input configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Change only this path if you keep the CSV somewhere else.
# Colab users may mount Drive first and set DATA_PATH accordingly.
DATA_PATH = Path("data/online_retail_II.csv")
DRIVE_PATH = Path("/content/drive/MyDrive/BTT_Amazon1A_Studio/online_retail_II.csv")

if not DATA_PATH.is_file() and DRIVE_PATH.is_file():
    DATA_PATH = DRIVE_PATH

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Upload the repository's data/online_retail_II.csv "
        "to your working directory, or change DATA_PATH. In Colab, mount Google Drive "
        "first if using DRIVE_PATH."
    )

raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded {len(raw):,} transaction rows from {DATA_PATH}")
print("Available columns:", raw.columns.tolist())

## 2. Prepare transactions consistently with the EDA notebook

The existing EDA drops missing customer IDs, deduplicates rows, treats invoice IDs beginning with `C` as cancellations, and groups customer/product/month transactions to net returns. It also reconstructs dates without their time-of-day; here we explicitly normalize timestamps to midnight to match. Invalid numeric/date rows are excluded with counts reported rather than silently treated as purchases.

In [ ]:
required = {"Invoice", "StockCode", "Quantity", "Price", "Customer ID", "InvoiceDate"}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f"Missing required CSV columns: {sorted(missing)}")

transactions = raw.dropna(subset=["Customer ID"]).copy()
transactions["InvoiceDate"] = pd.to_datetime(transactions["InvoiceDate"], errors="coerce")
for col in ["Quantity", "Price"]:
    transactions[col] = pd.to_numeric(transactions[col], errors="coerce")

before_validity = len(transactions)
transactions = transactions.dropna(subset=["InvoiceDate", "Quantity", "Price", "Invoice", "StockCode"])
print(f"Dropped {len(raw) - before_validity:,} rows without Customer ID")
print(f"Dropped {before_validity - len(transactions):,} rows with missing/invalid required values")

# Customer IDs are numeric in the original EDA. Match its string representation.
ids = pd.to_numeric(transactions["Customer ID"], errors="coerce")
if ids.isna().any() or (ids % 1 != 0).any():
    raise ValueError("Customer ID contains non-integer/non-numeric values; inspect before proceeding.")
transactions["Customer ID"] = ids.astype("int64").astype(str)
transactions["StockCode"] = transactions["StockCode"].astype(str)
transactions["Invoice"] = transactions["Invoice"].astype(str)
transactions["InvoiceDate"] = transactions["InvoiceDate"].dt.normalize()
transactions = transactions.drop_duplicates().copy()

transactions["SnapshotMonth"] = transactions["InvoiceDate"].dt.to_period("M")
transactions["Revenue"] = transactions["Quantity"] * transactions["Price"]
transactions["IsCancellation"] = transactions["Invoice"].str.startswith("C")
print(f"Transactions after preprocessing: {len(transactions):,}")

## 3. Reproduce the existing monthly measures

The grouping below mirrors `EDA.ipynb` rather than substituting an alternative definition of orders or items. This makes comparisons with existing work interpretable. In particular, `month_orders` is the number of distinct **first invoice IDs retained after netting**; it should not be presented as an exact count of original completed invoices.

In [ ]:
# Preserve the EDA's customer/product/month netting approach.
netted = (
    transactions.groupby(["Customer ID", "StockCode", "SnapshotMonth"], as_index=False)
    .agg(
        Quantity=("Quantity", "sum"),
        Invoice=("Invoice", "first"),
        Revenue=("Revenue", "sum"),
        CancelCount=("IsCancellation", "sum"),
    )
)
monthly_data = (
    netted.groupby(["Customer ID", "SnapshotMonth"], as_index=False)
    .agg(
        month_orders=("Invoice", "nunique"),
        month_revenue=("Revenue", "sum"),
        month_quantity=("Quantity", "sum"),
        month_diff_items=("StockCode", "nunique"),
        month_cancellations=("CancelCount", "sum"),
    )
)

# Include every month from the first observed transaction to the dataset end,
# including months without purchases. Do not filter by churn label here.
first_month = transactions.groupby("Customer ID")["SnapshotMonth"].min()
last_month = transactions["SnapshotMonth"].max()
calendar = pd.concat(
    [pd.DataFrame({"Customer ID": customer_id,
                   "SnapshotMonth": pd.period_range(start=first, end=last_month, freq="M")})
     for customer_id, first in first_month.items()],
    ignore_index=True,
)
monthly = calendar.merge(monthly_data, on=["Customer ID", "SnapshotMonth"], how="left", validate="one_to_one")
count_cols = ["month_orders", "month_revenue", "month_quantity", "month_diff_items", "month_cancellations"]
monthly[count_cols] = monthly[count_cols].fillna(0)
monthly = monthly.sort_values(["Customer ID", "SnapshotMonth"]).reset_index(drop=True)
print(f"Customer-month observations: {len(monthly):,}; customers: {monthly['Customer ID'].nunique():,}")

## 4. Compute recency and average order value

Recency uses only **positive-quantity invoices** (as in the existing EDA), and only orders on or before the snapshot date. For AOV, months with no orders are assigned `0` for the *initial drift heuristic*; the percentage change is left undefined if the previous AOV was nonpositive. Negative net revenue remains possible after cancellations.

In [ ]:
# One row per positive-quantity invoice; this matches EDA's recency basis.
orders = (
    transactions.loc[transactions["Quantity"] > 0, ["Customer ID", "Invoice", "InvoiceDate"]]
    .groupby(["Customer ID", "Invoice"], as_index=False)["InvoiceDate"].min()
    .rename(columns={"InvoiceDate": "last_order"})
    .sort_values("last_order")
)
if orders.empty:
    raise ValueError("No positive-quantity invoices; recency cannot be calculated.")

monthly["snapshot_end"] = monthly["SnapshotMonth"].dt.to_timestamp(how="end").dt.normalize()
monthly = pd.merge_asof(
    monthly.sort_values("snapshot_end"),
    orders[["Customer ID", "last_order"]].sort_values("last_order"),
    left_on="snapshot_end", right_on="last_order",
    by="Customer ID", direction="backward", allow_exact_matches=True,
)
monthly["recency_days"] = (monthly["snapshot_end"] - monthly["last_order"]).dt.days
monthly["avg_order_value"] = (
    monthly["month_revenue"] / monthly["month_orders"].replace(0, np.nan)
).fillna(0)
monthly = monthly.sort_values(["Customer ID", "SnapshotMonth"]).reset_index(drop=True)

assert monthly["recency_days"].dropna().ge(0).all(), "Found a future-dated order in recency calculation"
print("Rows with no prior positive-quantity order:", monthly["recency_days"].isna().sum())

## 5. Previous-month changes and initial drift score

Changes are current month minus immediately preceding calendar month. Negative order/revenue/item/AOV changes and positive recency changes are warning signals. Missing previous-month history receives **no score** (not a false zero). Percentage changes with a previous value of zero or less remain undefined. Score range: 0–5. Labels are illustrative, not empirically calibrated.

In [ ]:
drift_features = monthly.copy()
measures = ["month_orders", "month_revenue", "month_diff_items", "avg_order_value", "recency_days"]
groups = drift_features.groupby("Customer ID", sort=False)

for name in measures:
    drift_features[f"prev_{name}"] = groups[name].shift(1)
    drift_features[f"change_{name}"] = drift_features[name] - drift_features[f"prev_{name}"]

for name in ["month_orders", "month_revenue", "month_diff_items", "avg_order_value"]:
    previous = drift_features[f"prev_{name}"]
    drift_features[f"pct_change_{name}"] = (
        100 * drift_features[f"change_{name}"] / previous.where(previous > 0)
    )

# Explicitly verify that every available predecessor is the prior CALENDAR month.
previous_month = groups["SnapshotMonth"].shift(1)
valid_previous_month = previous_month.isna() | (
    drift_features["SnapshotMonth"] == (previous_month + 1)
)
assert valid_previous_month.all(), "Nonconsecutive customer-month rows found"

has_previous = previous_month.notna()
# Recency must exist in both months to score all five components.
has_recency_pair = drift_features["recency_days"].notna() & drift_features["prev_recency_days"].notna()

warning_columns = {
    "orders_declined": drift_features["change_month_orders"] < 0,
    "revenue_declined": drift_features["change_month_revenue"] < 0,
    "unique_items_declined": drift_features["change_month_diff_items"] < 0,
    "aov_declined": drift_features["change_avg_order_value"] < 0,
    "recency_increased": drift_features["change_recency_days"] > 0,
}
for column, condition in warning_columns.items():
    drift_features[column] = condition.where(has_previous & has_recency_pair).astype("boolean")

score_columns = list(warning_columns)
drift_features["drift_score"] = drift_features[score_columns].sum(axis=1, min_count=5).astype("Int64")
drift_features["drift_level"] = pd.cut(
    drift_features["drift_score"].astype("float"),
    bins=[-0.5, 1.5, 3.5, 5.5], labels=["Low", "Moderate", "High"],
)
print("Scored customer-months:", drift_features["drift_score"].notna().sum())
print(drift_features["drift_level"].value_counts(dropna=False))

## 6. Validation and results

These checks confirm uniqueness, chronological ordering, missing first-month scores, and score bounds. They do **not** replace an end-to-end run against the complete repository dataset or validation against future churn outcomes.

In [ ]:
assert not drift_features.duplicated(["Customer ID", "SnapshotMonth"]).any()
assert drift_features["drift_score"].dropna().between(0, 5).all()
assert drift_features.loc[~has_previous, "drift_score"].isna().all()
assert (drift_features["snapshot_end"] >= drift_features["last_order"]).fillna(True).all()

show_cols = [
    "Customer ID", "SnapshotMonth", "month_orders", "month_revenue",
    "month_diff_items", "avg_order_value", "recency_days",
    "change_month_orders", "change_month_revenue", "change_month_diff_items",
    "change_avg_order_value", "change_recency_days", "drift_score", "drift_level",
]
display(drift_features[show_cols].head(20))

counts = drift_features["drift_level"].value_counts().reindex(["Low", "Moderate", "High"], fill_value=0)
ax = counts.plot.bar(figsize=(7, 4), title="Initial behavioral drift score distribution")
ax.set(xlabel="Drift level", ylabel="Customer-month observations")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print("Validation checks passed.")

## 7. Optional export (disabled by default)

Keep this disabled until you have reviewed the results. The export creates a **separate** CSV and never overwrites `feat.csv`. This output includes customer-months that may lack valid 90-day churn labels; downstream supervised models must join to eligible labeled rows and perform time-aware evaluation.

In [ ]:
EXPORT_CSV = False  # Set True only after reviewing the results.
OUTPUT_PATH = Path("behavioral_drift_features.csv")

if EXPORT_CSV:
    if OUTPUT_PATH.exists():
        raise FileExistsError(f"Refusing to overwrite existing file: {OUTPUT_PATH}")
    drift_features.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved {len(drift_features):,} rows to {OUTPUT_PATH.resolve()}")
else:
    print("No files written. Set EXPORT_CSV = True to create a separate CSV.")

## Notes for your team

- **Task #6 covered:** month-over-month change in purchase frequency (`month_orders`), revenue, unique items, average order value, and recency; plus an initial drift score and visualization.
- **No model impact yet:** Logistic Regression, Random Forest, and XGBoost read their existing `feat.csv` and explicit `FEATURES` lists. They will not use these features unless your team later joins them and retrains.
- **EDA limitations preserved:** Monthly invoice counts and netting logic are intentionally consistent with `EDA.ipynb`; the EDA's order count is not a definitive unique-purchase count. AOV is therefore an EDA-compatible proxy.
- **Recency caveat:** A rise in recency across two calendar month-ends may simply reflect the passage of time; it is an initial heuristic signal, not proof of increased churn risk.
- **No churn label is calculated here:** Avoids changing your team's existing churn definition. November work can evaluate whether these features improve out-of-time churn prediction and early-warning horizon.
- **Execution environment:** Run with the repository CSV available locally, or mount Drive and update `DATA_PATH`. The notebook does not require the existing EDA notebook to have been run first.